In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r"C:\Users\satis\Downloads\Projects\Amazon Sale Report.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128976 entries, 0 to 128975
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128976 non-null  int64  
 1   Order ID            128976 non-null  object 
 2   Date                128976 non-null  object 
 3   Status              128976 non-null  object 
 4   Fulfilment          128976 non-null  object 
 5   Sales Channel       128976 non-null  object 
 6   ship-service-level  128976 non-null  object 
 7   Category            128976 non-null  object 
 8   Size                128976 non-null  object 
 9   Courier Status      128976 non-null  object 
 10  Qty                 128976 non-null  int64  
 11  currency            121176 non-null  object 
 12  Amount              121176 non-null  float64
 13  ship-city           128941 non-null  object 
 14  ship-state          128941 non-null  object 
 15  ship-postal-code    128941 non-nul

In [4]:
df = df.drop(['New','PendingS'], axis = 1)

In [5]:
pd.isnull(df).sum()

index                     0
Order ID                  0
Date                      0
Status                    0
Fulfilment                0
Sales Channel             0
ship-service-level        0
Category                  0
Size                      0
Courier Status            0
Qty                       0
currency               7800
Amount                 7800
ship-city                35
ship-state               35
ship-postal-code         35
ship-country             35
B2B                       0
fulfilled-by          89713
dtype: int64

In [6]:
# Remove the null values
df = df.dropna(subset = ['Amount','ship-city', 'ship-state', 'ship-postal-code','ship-country'])

In [7]:
pd.crosstab(
    df['Fulfilment'],
    df['fulfilled-by'],
    dropna=False
)

fulfilled-by,Easy Ship,NaN
Fulfilment,,
Amazon,0,83629
Merchant,37514,0


In [8]:
# The missing values in fulfilled-by are not data-quality issues. They represent merchant-fulfilled orders. 
# fulfilled-by is redundant with Fulfilment, so dropping that column won't affect the analysis

In [9]:
df.drop('fulfilled-by', axis=1, inplace=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 121143 entries, 0 to 128975
Data columns (total 18 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               121143 non-null  int64  
 1   Order ID            121143 non-null  object 
 2   Date                121143 non-null  object 
 3   Status              121143 non-null  object 
 4   Fulfilment          121143 non-null  object 
 5   Sales Channel       121143 non-null  object 
 6   ship-service-level  121143 non-null  object 
 7   Category            121143 non-null  object 
 8   Size                121143 non-null  object 
 9   Courier Status      121143 non-null  object 
 10  Qty                 121143 non-null  int64  
 11  currency            121143 non-null  object 
 12  Amount              121143 non-null  float64
 13  ship-city           121143 non-null  object 
 14  ship-state          121143 non-null  object 
 15  ship-postal-code    121143 non-null  fl

In [11]:
# Change datatype

In [12]:
df['ship-postal-code'] = df['ship-postal-code'].astype('int')

In [13]:
df['ship-postal-code'].dtype

dtype('int64')

In [14]:
type('Date')

str

In [23]:
df['Date'] = pd.to_datetime(df['Date'], format='mixed')

In [26]:
df['Date'].dtype

dtype('<M8[ns]')

In [29]:
df.describe()

,index,Date,Qty,Amount,ship-postal-code
count,121143.000000,121143,121143.000000,121143.000000,121143.000000
mean,64486.312655,2022-05-12 12:11:22.182214144,0.961252,648.576874,463623.724507
min,0.000000,2022-03-31 00:00:00,0.000000,0.000000,110001.000000
25%,32294.500000,2022-04-20 00:00:00,1.000000,449.000000,382421.000000
50%,64477.000000,2022-05-10 00:00:00,1.000000,605.000000,500032.000000
75%,96682.500000,2022-06-04 00:00:00,1.000000,788.000000,600020.000000
max,128974.000000,2022-06-29 00:00:00,8.000000,5584.000000,989898.000000
std,37220.415404,NaN,0.214276,281.196896,191301.588170


In [30]:
df.describe(include = 'object')

,Order ID,Status,Fulfilment,Sales Channel,ship-service-level,Category,Size,Courier Status,currency,ship-city,ship-state,ship-country
count,121143,121143,121143,121143,121143,121143,121143,121143,121143,121143,121143,121143
unique,112861,12,2,1,2,9,11,3,1,8697,68,1
top,171-5057375-2831560,Shipped,Amazon,Amazon.in,Expedited,T-shirt,M,Shipped,INR,BENGALURU,MAHARASHTRA,IN
freq,12,77589,83629,121143,82713,47038,20965,109458,121143,10675,21084,121143


In [31]:
df['Year'] = df['Date'].dt.year

df['Month'] = df['Date'].dt.month_name()

df['Month_No'] = df['Date'].dt.month

df['Day'] = df['Date'].dt.day

In [32]:
df = df.sort_values('Month_No')

In [33]:
# Total Revenue
df['Amount'].sum()

np.float64(78570548.25)

In [34]:
# Total Orders
df['Order ID'].nunique()

112861

In [35]:
# Total Quantity Sold
df['Qty'].sum()

np.int64(116449)

In [36]:
# Top Categories

category_sales = df.groupby('Category')['Amount'].sum().sort_values(ascending = False)
category_sales

Category
T-shirt     39197808.65
Shirt       21289304.08
Blazzer     11214369.12
Trousers     5344813.30
Perfume       789419.66
Wallet        458408.18
Socks         150757.50
Shoes         124752.76
Watch            915.00
Name: Amount, dtype: float64

In [37]:
# Top States by Revenue

state_sales = df.groupby('ship-state')['Amount'].sum().sort_values(ascending = False)

In [38]:
state_sales.head(10)

ship-state
MAHARASHTRA       13340333.05
KARNATAKA         10480694.22
TELANGANA          6915018.08
UTTAR PRADESH      6823947.08
TAMIL NADU         6519182.30
DELHI              4232738.97
KERALA             3823559.58
WEST BENGAL        3507212.82
ANDHRA PRADESH     3217859.86
HARYANA            2880355.99
Name: Amount, dtype: float64

In [39]:
# Top Cities

city_sales = df.groupby('ship-city')['Amount'].sum().sort_values(ascending = False)
city_sales.head()

ship-city
BENGALURU    6845390.65
HYDERABAD    4946394.25
MUMBAI       3701843.04
NEW DELHI    3612512.78
CHENNAI      3103415.74
Name: Amount, dtype: float64

In [40]:
# Fulfillment Analysis

compare = df.groupby('Fulfilment')['Amount'].sum()
compare.head()

Fulfilment
Amazon      54315723.00
Merchant    24254825.25
Name: Amount, dtype: float64

In [41]:
monthly_sales = df.groupby('Month')['Amount'].sum()

In [42]:
monthly_sales.head()

Month
April    28827790.27
June     23421223.38
March      101683.85
May      26219850.75
Name: Amount, dtype: float64

In [43]:
total_revenue = df['Amount'].sum()
total_revenue

np.float64(78570548.25)

In [44]:
# Cancellation Rate

cancelled = df[df['Status'].str.contains('Cancelled', na=False)]

rate = (len(cancelled)/len(df))*100
rate

8.8804140561155

In [45]:
df.to_csv(r"C:\Users\satis\Downloads\Projects\Sales_cleaned2.csv", index=False)

In [46]:
df.head()

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Category,Size,Courier Status,...,Amount,ship-city,ship-state,ship-postal-code,ship-country,B2B,Year,Month,Month_No,Day
48958,48957,405-9330131-9228332,2022-03-31,Shipped,Amazon,Amazon.in,Expedited,Shirt,3XL,Shipped,...,293.00,AHMEDABAD,Gujarat,380055,IN,False,2022,March,3,31
48989,48988,405-9034577-8866749,2022-03-31,Shipped,Amazon,Amazon.in,Expedited,Shirt,3XL,Shipped,...,353.00,CHANDIGARH,CHANDIGARH,160047,IN,False,2022,March,3,31
48990,48989,407-2352301-8497951,2022-03-31,Shipped,Amazon,Amazon.in,Expedited,Shirt,XXL,Shipped,...,499.00,LUCKNOW,UTTAR PRADESH,226006,IN,False,2022,March,3,31
48991,48990,407-7115918-3595555,2022-03-31,Cancelled,Merchant,Amazon.in,Standard,Blazzer,M,On the Way,...,724.76,VADAKARA,KERALA,673101,IN,False,2022,March,3,31
48992,48991,402-6785096-8802758,2022-03-31,Shipped,Amazon,Amazon.in,Expedited,Shirt,XXL,Shipped,...,481.00,AGRA,UTTAR PRADESH,282002,IN,False,2022,March,3,31
